In [3]:
# =========================================
# LwF on EuroSAT (torchvision) - raw/direct implementation
# =========================================
# Paper: Learning without Forgetting (Li & Hoiem, ECCV 2016 / arXiv:1606.09282)

import os
import json
import copy
import random
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import EuroSAT
from torchvision.models import resnet18, ResNet18_Weights


# -----------------------------
# Config
# -----------------------------
ROOT = "./data"
SAVE_PATH = "./lwf_eurosat_results.json"

SEEDS = [1, 2, 3]
QUICK_RUN = False

BS = 64
NW = 4
IMG_SIZE = 224

# LwF hyperparams (paper-style choices)
T_DISTILL = 2.0
LAMBDA_OLD = 1.0
WEIGHT_DECAY = 5e-4
MOMENTUM = 0.9

# Epochs
EPOCHS_A = 20 if not QUICK_RUN else 2
EPOCHS_B_WARMUP = 5 if not QUICK_RUN else 1
EPOCHS_B = 20 if not QUICK_RUN else 2

# Learning rates
LR_A = 0.01
LR_B_WARMUP = 0.01
LR_B = 0.005

# Task split: A = first 5 classes, B = last 5 classes
NUM_CLASSES_TOTAL = 10
A_CLASSES = list(range(0, 5))
B_CLASSES = list(range(5, 10))


# -----------------------------
# Utilities
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def save_progress(meta: dict):
    with open(SAVE_PATH, "w") as f:
        json.dump(meta, f, indent=2)


def stratified_split_indices(targets: List[int], seed: int,
                             train_ratio=0.6, val_ratio=0.2):
    rng = np.random.RandomState(seed)
    targets = np.array(targets)
    train_idx, val_idx, test_idx = [], [], []

    for c in np.unique(targets):
        idx = np.where(targets == c)[0]
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        train_idx.extend(idx[:n_train].tolist())
        val_idx.extend(idx[n_train:n_train+n_val].tolist())
        test_idx.extend(idx[n_train+n_val:].tolist())

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx


def filter_indices_by_classes(targets: List[int], indices: List[int], cls_set: List[int]):
    cls_set = set(cls_set)
    return [i for i in indices if targets[i] in cls_set]


class MappedSubset(Dataset):
    """
    Wrap a subset and remap labels:
      - for task A: global 0..4 -> local 0..4
      - for task B: global 5..9 -> local 0..4 (subtract 5)
      - for CIL eval: keep global labels
    """
    def __init__(self, base_ds, indices, label_map=None):
        self.base_ds = base_ds
        self.indices = indices
        self.label_map = label_map

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x, y = self.base_ds[self.indices[i]]
        if self.label_map is not None:
            y = self.label_map[y]
        return x, y


# -----------------------------
# Model: shared backbone + task-specific heads
# -----------------------------
class LwFResNet18(nn.Module):
    def __init__(self, num_a=5, num_b=5):
        super().__init__()
        base = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        in_features = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base
        self.head_a = nn.Linear(in_features, num_a)  # old task
        self.head_b = nn.Linear(in_features, num_b)  # new task

    def forward_features(self, x):
        return self.backbone(x)

    def forward_a(self, x):
        f = self.forward_features(x)
        return self.head_a(f)

    def forward_b(self, x):
        f = self.forward_features(x)
        return self.head_b(f)

    def forward_cil_logits(self, x):
        # CIL logits over 10 classes = [A(0..4), B(5..9)]
        f = self.forward_features(x)
        la = self.head_a(f)
        lb = self.head_b(f)
        return torch.cat([la, lb], dim=1)


# -----------------------------
# Distillation loss (paper Eq. for old-task KD)
# -----------------------------
def distill_ce_loss(student_logits, teacher_logits, T=2.0):
    # Cross-entropy between softened teacher and student distributions
    p_teacher = F.softmax(teacher_logits / T, dim=1)
    log_p_student = F.log_softmax(student_logits / T, dim=1)
    loss = -(p_teacher * log_p_student).sum(dim=1).mean()
    return loss


# -----------------------------
# Train / Eval helpers
# -----------------------------
def train_task_a(model, loader, device, epochs, lr):
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model.forward_a(x)
            loss = F.cross_entropy(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


def warmup_task_b(model, loader_b, device, epochs, lr):
    # Freeze shared + old head; train only new head
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.head_a.parameters():
        p.requires_grad = False
    for p in model.head_b.parameters():
        p.requires_grad = True

    model.train()
    optimizer = torch.optim.SGD(model.head_b.parameters(), lr=lr,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    for _ in range(epochs):
        for x, yb in loader_b:
            x, yb = x.to(device), yb.to(device)
            logits_b = model.forward_b(x)
            loss_new = F.cross_entropy(logits_b, yb)

            optimizer.zero_grad()
            loss_new.backward()
            optimizer.step()


def joint_train_task_b_lwf(model, teacher_old, loader_b, device, epochs, lr, T=2.0, lam_old=1.0):
    # Unfreeze all (paper joint optimization step)
    for p in model.parameters():
        p.requires_grad = True

    model.train()
    teacher_old.eval()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

    for _ in range(epochs):
        for x, yb in loader_b:
            x, yb = x.to(device), yb.to(device)

            # New-task CE
            logits_b = model.forward_b(x)
            loss_new = F.cross_entropy(logits_b, yb)

            # Old-task distillation on same new-task images
            with torch.no_grad():
                t_old = teacher_old.forward_a(x)  # recorded old response surrogate
            s_old = model.forward_a(x)
            loss_old = distill_ce_loss(s_old, t_old, T=T)

            loss = loss_new + lam_old * loss_old

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


@torch.no_grad()
def eval_taskaware_a(model, loader, device):
    model.eval()
    n, c = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model.forward_a(x)
        pred = logits.argmax(dim=1)
        c += (pred == y).sum().item()
        n += y.numel()
    return 100.0 * c / max(1, n)


@torch.no_grad()
def eval_taskaware_b(model, loader, device):
    model.eval()
    n, c = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model.forward_b(x)
        pred = logits.argmax(dim=1)
        c += (pred == y).sum().item()
        n += y.numel()
    return 100.0 * c / max(1, n)


@torch.no_grad()
def eval_cil(model, loader_global_labels, device):
    model.eval()
    n, c = 0, 0
    for x, y in loader_global_labels:
        x, y = x.to(device), y.to(device)
        logits10 = model.forward_cil_logits(x)
        pred = logits10.argmax(dim=1)
        c += (pred == y).sum().item()
        n += y.numel()
    return 100.0 * c / max(1, n)


def run_lwf_eurosat(seed, loaders, device):
    (
        train_A_loader,
        val_A_loader,
        train_B_loader,
        val_B_loader,
        test_A_taskaware_loader,
        test_B_taskaware_loader,
        test_A_cil_loader,
        test_B_cil_loader,
        test_joint_cil_loader,
    ) = loaders

    set_seed(seed)
    model = LwFResNet18(num_a=5, num_b=5).to(device)

    # Stage A
    train_task_a(model, train_A_loader, device, EPOCHS_A, LR_A)
    acc_A_init = eval_taskaware_a(model, test_A_taskaware_loader, device)

    # Teacher snapshot before learning B
    teacher_old = copy.deepcopy(model).to(device)
    teacher_old.eval()
    for p in teacher_old.parameters():
        p.requires_grad = False

    # Stage B warm-up + joint LwF
    warmup_task_b(model, train_B_loader, device, EPOCHS_B_WARMUP, LR_B_WARMUP)
    joint_train_task_b_lwf(
        model, teacher_old, train_B_loader, device,
        EPOCHS_B, LR_B, T=T_DISTILL, lam_old=LAMBDA_OLD
    )

    # Final evals
    acc_A_final_taskaware = eval_taskaware_a(model, test_A_taskaware_loader, device)
    acc_B_final_taskaware = eval_taskaware_b(model, test_B_taskaware_loader, device)

    acc_A_final_cil = eval_cil(model, test_A_cil_loader, device)
    acc_B_final_cil = eval_cil(model, test_B_cil_loader, device)
    acc_joint_cil = eval_cil(model, test_joint_cil_loader, device)

    # Derived metrics
    retention_taskaware = 100.0 * (acc_A_final_taskaware / max(1e-8, acc_A_init))
    retention_cil = 100.0 * (acc_A_final_cil / max(1e-8, acc_A_init))

    forgetting_taskaware = acc_A_init - acc_A_final_taskaware
    forgetting_cil = acc_A_init - acc_A_final_cil

    bwt_taskaware = acc_A_final_taskaware - acc_A_init
    bwt_cil = acc_A_final_cil - acc_A_init

    return {
        "acc_A_init": acc_A_init,
        "acc_A_final_taskaware": acc_A_final_taskaware,
        "acc_B_final_taskaware": acc_B_final_taskaware,
        "acc_A_final_cil": acc_A_final_cil,
        "acc_B_final_cil": acc_B_final_cil,
        "acc_joint_cil": acc_joint_cil,
        "retention_taskaware": retention_taskaware,
        "retention_cil": retention_cil,
        "forgetting_taskaware": forgetting_taskaware,
        "forgetting_cil": forgetting_cil,
        "bwt_taskaware": bwt_taskaware,
        "bwt_cil": bwt_cil,
    }


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # Transforms
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ])

    # Build two dataset objects (same data, different transforms)
    train_base = EuroSAT(root=ROOT, download=True, transform=train_tf)
    eval_base = EuroSAT(root=ROOT, download=True, transform=eval_tf)

    # EuroSAT labels
    targets = list(train_base.targets)

    results = []
    meta = {
        "timestamp": str(datetime.utcnow()),
        "dataset": "EuroSAT",
        "method": "LwF (raw, Li & Hoiem 2016)",
        "quick_run": QUICK_RUN,
        "epochs_A": EPOCHS_A,
        "epochs_B_warmup": EPOCHS_B_WARMUP,
        "epochs_B": EPOCHS_B,
        "temperature": T_DISTILL,
        "lambda_old": LAMBDA_OLD,
        "device": str(device),
        "seeds": SEEDS,
        "results": []
    }

    print("\n[1] Running raw EuroSAT LwF experiments...")
    try:
        for s in SEEDS:
            print(f" -> Seed {s}...")
            set_seed(s)

            # Stratified split (global)
            train_idx, val_idx, test_idx = stratified_split_indices(targets, seed=s)

            # A/B class filtering per split
            train_A_idx = filter_indices_by_classes(targets, train_idx, A_CLASSES)
            val_A_idx = filter_indices_by_classes(targets, val_idx, A_CLASSES)
            test_A_idx = filter_indices_by_classes(targets, test_idx, A_CLASSES)

            train_B_idx = filter_indices_by_classes(targets, train_idx, B_CLASSES)
            val_B_idx = filter_indices_by_classes(targets, val_idx, B_CLASSES)
            test_B_idx = filter_indices_by_classes(targets, test_idx, B_CLASSES)

            # Label maps for task-aware training/eval
            map_A = {c: i for i, c in enumerate(A_CLASSES)}        # 0..4 -> 0..4
            map_B = {c: i for i, c in enumerate(B_CLASSES)}        # 5..9 -> 0..4

            loaders = (
                DataLoader(MappedSubset(train_base, train_A_idx, label_map=map_A), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=(NW > 0)),
                DataLoader(MappedSubset(eval_base, val_A_idx, label_map=map_A), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),
                DataLoader(MappedSubset(train_base, train_B_idx, label_map=map_B), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=(NW > 0)),
                DataLoader(MappedSubset(eval_base, val_B_idx, label_map=map_B), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),
                DataLoader(MappedSubset(eval_base, test_A_idx, label_map=map_A), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),   # task-aware A
                DataLoader(MappedSubset(eval_base, test_B_idx, label_map=map_B), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),   # task-aware B
                DataLoader(MappedSubset(eval_base, test_A_idx, label_map=None), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),    # CIL A
                DataLoader(MappedSubset(eval_base, test_B_idx, label_map=None), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),    # CIL B
                DataLoader(MappedSubset(eval_base, test_A_idx + test_B_idx, label_map=None), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),  # CIL Joint
            )

            r = run_lwf_eurosat(
                seed=s,
                loaders=loaders,
                device=device
            )

            results.append(r)
            meta["results"].append({"seed": s, **r})
            save_progress(meta)

            print(
                f"    ↳ Seed {s}: "
                f"A_init(TA)={r['acc_A_init']:.2f} | "
                f"A_final(TA)={r['acc_A_final_taskaware']:.2f} | "
                f"B_final(TA)={r['acc_B_final_taskaware']:.2f} | "
                f"A_final(CIL)={r['acc_A_final_cil']:.2f} | "
                f"B_final(CIL)={r['acc_B_final_cil']:.2f} | "
                f"Joint_CIL={r['acc_joint_cil']:.2f}"
            )

    except KeyboardInterrupt:
        print("\n⛔ Interrupted. Partial results saved to:", SAVE_PATH)
        save_progress(meta)
        return

    if results:
        print("\n" + "=" * 100)
        print("SUMMARY (Mean ± Std)")
        print("=" * 100)

        for k, label in [
            ("acc_A_init", "A Init (Task-aware)"),
            ("acc_A_final_taskaware", "A Final (Task-aware)"),
            ("acc_B_final_taskaware", "B Final (Task-aware)"),
            ("acc_A_final_cil", "A Final (CIL)"),
            ("acc_B_final_cil", "B Final (CIL)"),
            ("acc_joint_cil", "Joint Test CIL"),
        ]:
            vals = [r[k] for r in results]
            print(f"{label:26}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

        for k, label in [
            ("retention_taskaware", "Retention (Task-aware)"),
            ("retention_cil", "Retention (CIL)"),
            ("forgetting_taskaware", "Forgetting (Task-aware)"),
            ("forgetting_cil", "Forgetting (CIL)"),
            ("bwt_taskaware", "BWT (Task-aware)"),
            ("bwt_cil", "BWT (CIL)")
        ]:
            vals = [r[k] for r in results]
            print(f"{label:26}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

        print("=" * 100)
        print(f"Saved run log to: {SAVE_PATH}")


if __name__ == "__main__":
    main()

Device: cuda

[1] Running raw EuroSAT LwF experiments...
 -> Seed 1...


/tmp/ipykernel_55/1814869135.py:367: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": str(datetime.utcnow()),


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 203MB/s]


    ↳ Seed 1: A_init(TA)=98.93 | A_final(TA)=99.00 | B_final(TA)=99.69 | A_final(CIL)=90.32 | B_final(CIL)=96.65 | Joint_CIL=93.37
 -> Seed 2...
    ↳ Seed 2: A_init(TA)=99.46 | A_final(TA)=99.36 | B_final(TA)=99.73 | A_final(CIL)=87.43 | B_final(CIL)=98.46 | Joint_CIL=92.74
 -> Seed 3...
    ↳ Seed 3: A_init(TA)=99.46 | A_final(TA)=99.54 | B_final(TA)=99.65 | A_final(CIL)=91.75 | B_final(CIL)=97.62 | Joint_CIL=94.57

SUMMARY (Mean ± Std)
A Init (Task-aware)       : 99.29 ± 0.25
A Final (Task-aware)      : 99.30 ± 0.22
B Final (Task-aware)      : 99.69 ± 0.03
A Final (CIL)             : 89.83 ± 1.80
B Final (CIL)             : 97.58 ± 0.74
Joint Test CIL            : 93.56 ± 0.76
Retention (Task-aware)    : 100.01 ± 0.08
Retention (CIL)           : 90.48 ± 1.87
Forgetting (Task-aware)   : -0.01 ± 0.08
Forgetting (CIL)          : 9.45 ± 1.86
BWT (Task-aware)          : 0.01 ± 0.08
BWT (CIL)                 : -9.45 ± 1.86
Saved run log to: ./lwf_eurosat_results.json


In [ ]:
'''
A Init (Task-aware)       : 93.75 ± 0.49
A Final (Task-aware)      : 93.07 ± 0.20
B Final (Task-aware)      : 94.56 ± 0.81
A Final (CIL)             : 69.45 ± 0.56
B Final (CIL)             : 85.62 ± 0.93
Joint Test CIL            : 77.57 ± 0.70
Retention (Task-aware)    : 99.23 ± 0.68
Retention (CIL)           : 72.43 ± 0.26
Forgetting (Task-aware)   : 0.68 ± 0.63
Forgetting (CIL)          : 24.30 ± 0.15
BWT (Task-aware)          : -0.68 ± 0.63
BWT (CIL)                 : -24.30 ± 0.15

'''